# 04 - Restoration (Part 3: enhance distorted images, re-measure)

Applies each distortion's course-grounded restoration counterpart (median
filter / frequency-domain deconvolution / bilateral filtering + interpolation)
and re-runs all 4 tasks, mirroring 03's sweep structure so distorted vs.
restored performance is directly comparable. The clean baseline (03's
zero-distortion reference point) lives in `02_clean_baseline.csv`.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
from ultralytics import YOLO

from ipproj import config
from ipproj.datasets import kitti, kitti_flow
from ipproj.datasets.kitti import read_image
from ipproj.datasets.kitti_flow import read_kitti_flow_png
from ipproj.datasets.materialize import materialize_transformed
from ipproj.tasks import feature_matching, optical_flow, object_detection, semantic_segmentation
from ipproj.distortions import REGISTRY as DISTORTIONS
from ipproj.restoration import REGISTRY as RESTORATIONS
from ipproj.viz.plotting import plot_before_after, plot_metric_vs_intensity, save_figure

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()
flow_splits = kitti_flow.load_optical_flow_subset()

checkpoint_path = (config.CHECKPOINT_ROOT / "yolo_clean_baseline_path.txt").read_text().strip()
yolo_model = YOLO(checkpoint_path)
segformer_model, segformer_processor = semantic_segmentation.load_pretrained()

distortion_results = pd.read_csv(config.RESULTS_ROOT / "03_distortions.csv")

## Before/after: distorted vs. restored (strongest intensity)

In [ ]:
def make_restore_fn(distortion_module, level, restoration_module):
    if restoration_module.NAME == "motion_blur":
        return lambda img: restoration_module.restore(distortion_module.distort(img, level), kernel_size=distortion_module.LEVELS[level])
    return lambda img: restoration_module.restore(distortion_module.distort(img, level))


sample_image = read_image(detection_splits["test"][0].image_path)
for name, module in DISTORTIONS.items():
    level = len(module.LEVELS) - 1
    restore_fn = make_restore_fn(module, level, RESTORATIONS[name])
    distorted = module.distort(sample_image, level)
    restored = restore_fn(sample_image)
    fig = plot_before_after(distorted, restored, title_before=f"{name} distorted", title_after=f"{name} restored")
    save_figure(fig, f"04_restoration/before_after_{name}.png")

## Restoration sweep (mirrors 03's structure)

In [ ]:
restoration_results = []

for name, module in DISTORTIONS.items():
    restoration_module = RESTORATIONS[name]
    for level in range(len(module.LEVELS)):
        restore_fn = make_restore_fn(module, level, restoration_module)
        out_dir = config.RESTORED_ROOT / name / f"level_{level}"

        restored_detection = materialize_transformed(detection_splits["test"], restore_fn, out_dir / "detection")
        detection_metrics = object_detection.evaluate(yolo_model, restored_detection)

        restored_segmentation = materialize_transformed(segmentation_splits["test"], restore_fn, out_dir / "segmentation")
        segmentation_metrics = semantic_segmentation.evaluate(segformer_model, segformer_processor, restored_segmentation)

        match_accuracies = []
        for sample in detection_splits["test"][:20]:
            clean = read_image(sample.image_path)
            match_accuracies.append(feature_matching.match(clean, restore_fn(clean))[3])

        epe_values = []
        for sample in flow_splits["test"]:
            frame1 = read_image(sample.frame1_path)
            frame2 = read_image(sample.frame2_path)
            gt_flow, valid = read_kitti_flow_png(sample.flow_gt_path)
            epe_values.append(optical_flow.evaluate(frame1, restore_fn(frame2), gt_flow, valid))

        restoration_results.append({
            "distortion": name,
            "level": level,
            "match_accuracy": float(np.mean(match_accuracies)),
            "epe": float(np.mean(epe_values)),
            "map": float(detection_metrics["map"]),
            "mean_iou": float(segmentation_metrics.mean()),
        })

restoration_df = pd.DataFrame(restoration_results)
restoration_df.to_csv(config.RESULTS_ROOT / "04_restoration.csv", index=False)
restoration_df

## Distorted vs. restored comparison

In [ ]:
metric_labels = [("map", "mAP"), ("mean_iou", "mean IoU"), ("epe", "EPE"), ("match_accuracy", "match accuracy")]

for name in DISTORTIONS:
    distorted_subset = distortion_results[distortion_results["distortion"] == name].sort_values("level")
    restored_subset = restoration_df[restoration_df["distortion"] == name].sort_values("level")
    for metric_col, ylabel in metric_labels:
        fig = plot_metric_vs_intensity(
            distorted_subset["level"].tolist(),
            {"distorted": distorted_subset[metric_col].tolist(), "restored": restored_subset[metric_col].tolist()},
            xlabel="distortion intensity level", ylabel=ylabel, title=f"{name}: {ylabel} - distorted vs restored",
        )
        save_figure(fig, f"04_restoration/{name}_{metric_col}.png")